# Laplace-Beltrami Spectrum on 2D Domains

## Overview

The **spectrum of the Laplacian** encodes deep geometric information about a domain. The celebrated question "Can you hear the shape of a drum?" (Kac, 1966) asks whether the set of eigenvalues $\{\lambda_k\}$ of the Laplace operator uniquely determines the shape of a 2D region.

### The eigenvalue problem

Given a bounded domain $\Omega \subset \mathbb{R}^2$ with Dirichlet boundary conditions, we seek pairs $(\lambda, u)$ satisfying:
$$-\Delta u = \lambda u \quad \text{on } \Omega, \qquad u\big|_{\partial\Omega} = 0$$

The solutions form a discrete spectrum $0 < \lambda_1 \leq \lambda_2 \leq \lambda_3 \leq \cdots \to \infty$ with corresponding orthonormal eigenfunctions $\{u_k\}$ forming a basis for $L^2(\Omega)$.

### Analytical solution on a rectangle

For the rectangle $\Omega = (0, a) \times (0, b)$, the eigenfunctions are:
$$u_{mn}(x, y) = \sin\!\left(\frac{m\pi x}{a}\right)\sin\!\left(\frac{n\pi y}{b}\right), \quad m, n = 1, 2, 3, \ldots$$
with eigenvalues:
$$\lambda_{mn} = \pi^2\left(\frac{m^2}{a^2} + \frac{n^2}{b^2}\right)$$

### Finite difference discretization

On a uniform grid with spacing $h$, the 5-point stencil approximates $-\Delta$:
$$[-\Delta u]_{i,j} \approx \frac{1}{h^2}\bigl(4u_{ij} - u_{i-1,j} - u_{i+1,j} - u_{i,j-1} - u_{i,j+1}\bigr)$$
giving a sparse symmetric positive-definite matrix $L \in \mathbb{R}^{N^2 \times N^2}$ whose eigenpairs approximate those of the continuous problem.

### Weyl's asymptotic law

For large $k$, the eigenvalues satisfy:
$$\lambda_k \sim \frac{4\pi k}{|\Omega|}$$
where $|\Omega|$ is the area of the domain. This allows an estimate of the area from spectral data alone.

### What this notebook demonstrates

1. Analytical vs. numerical eigenmodes of a rectangle.
2. First 9 eigenmodes for a square domain, circle, L-shape, and ring.
3. Spectral comparison: eigenvalue sequences across shapes.
4. Weyl's law verification.

### Imports

We use `scipy.sparse` for efficient sparse matrix assembly and `scipy.sparse.linalg.eigsh` for computing the smallest eigenvalues of the symmetric positive-definite Laplacian matrix.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.sparse import diags, kron, eye
from scipy.sparse.linalg import eigsh
from matplotlib.colors import TwoSlopeNorm

### Finite difference Laplacian on a rectangular domain

We assemble the 5-point stencil matrix for $-\Delta$ on an $n \times n$ interior grid with Dirichlet BC. Using Kronecker products:
$$L = \frac{1}{h^2}(T_n \otimes I_n + I_n \otimes T_n)$$
where $T_n$ is the 1D tridiagonal matrix with $2$ on the diagonal and $-1$ on super/sub-diagonals. Interior nodes only, so boundary values $= 0$ are implicitly enforced.

In [2]:
def fd_laplacian_2d(n):
    """Assemble 5-point FD Laplacian on n x n interior grid, Dirichlet BC."""
    h = 1.0 / (n + 1)
    T = diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr')
    I = eye(n, format='csr')
    L = (kron(I, T) + kron(T, I)) / h**2
    return L, h

def compute_eigenmodes(L, n_modes=9):
    """Compute smallest n_modes eigenvalues and eigenvectors of L."""
    vals, vecs = eigsh(L, k=n_modes, which='SM', tol=1e-10)
    idx = np.argsort(vals)
    return vals[idx], vecs[:, idx]

# Test: rectangle with n x n interior points
n_grid = 40
L_rect, h_rect = fd_laplacian_2d(n_grid)
evals_num, evecs_num = compute_eigenmodes(L_rect, n_modes=9)
print('Numerical eigenvalues (rectangle):', np.round(evals_num[:6], 2))

Numerical eigenvalues (rectangle): [19.73 49.27 49.27 78.8  98.3  98.3 ]


/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_52434/1991806073.py:4: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
  T = diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr')


### Analytical vs. numerical eigenmodes on the unit square

For the unit square $\Omega = (0,1)^2$, the analytical eigenvalues are:
$$\lambda_{mn} = \pi^2(m^2 + n^2), \quad m,n \geq 1$$

The FD approximation converges to these values as $h \to 0$. We compare the first 9 eigenvalues and show the first eigenmode (fundamental mode, nodeless).

In [3]:
# Analytical eigenvalues for unit square
modes = [(m, n) for m in range(1, 6) for n in range(1, 6)]
lambda_analytic = sorted([np.pi**2 * (m**2 + n**2) for m, n in modes])[:9]

print('Analytical eigenvalues:', np.round(lambda_analytic[:6], 2))
print('Numerical eigenvalues: ', np.round(evals_num[:6], 2))
print('Relative errors:       ', np.round(np.abs(np.array(lambda_analytic[:6]) - evals_num[:6]) / np.array(lambda_analytic[:6]), 4))

# Analytical eigenfunctions on a fine grid
x = np.linspace(0, 1, n_grid + 2)[1:-1]
X, Y = np.meshgrid(x, x)

# First 9 modes sorted by eigenvalue
sorted_modes = sorted(modes, key=lambda mn: mn[0]**2 + mn[1]**2)[:9]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for idx, (ax, (m, n)) in enumerate(zip(axes.flat, sorted_modes)):
    u_analytic = np.sin(m * np.pi * X) * np.sin(n * np.pi * Y)
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    ax.contourf(X, Y, u_analytic, levels=20, cmap='RdBu_r', norm=norm)
    ax.contour(X, Y, u_analytic, levels=[0], colors='k', linewidths=0.8)
    ax.set_title(f'$u_{{{m},{n}}}$, $\\lambda={m**2+n**2}\\pi^2$', fontsize=9)
    ax.set_aspect('equal')
    ax.axis('off')
plt.suptitle('Analytical eigenmodes of the unit square', fontsize=12)
plt.tight_layout()
plt.savefig('rect_analytical.png', dpi=80, bbox_inches='tight')
plt.close()

Analytical eigenvalues: [19.74 49.35 49.35 78.96 98.7  98.7 ]
Numerical eigenvalues:  [19.73 49.27 49.27 78.8  98.3  98.3 ]
Relative errors:        [0.0005 0.0017 0.0017 0.002  0.004  0.004 ]


### Eigenmodes on different domain shapes

To compare spectra across shapes, we restrict the FD Laplacian to different domains by masking out grid points outside the domain. This is sometimes called the **embedding approach**: we build the full square Laplacian but retain only the interior points of the domain of interest.

We consider:
- **Square** $(0,1)^2$
- **Disk** of radius $0.45$ centered at $(0.5, 0.5)$
- **L-shape**: unit square minus the upper-right quarter
- **Ring**: annulus with inner radius $0.15$ and outer radius $0.45$

For each shape, we extract the sub-matrix corresponding to interior nodes and compute the first 9 eigenmodes.

In [4]:
n_s = 50  # grid size for shape experiments
x_s = np.linspace(0, 1, n_s + 2)[1:-1]
X_s, Y_s = np.meshgrid(x_s, x_s)
h_s = 1.0 / (n_s + 1)

def mask_laplacian(mask_2d, h):
    """
    Build FD Laplacian restricted to True cells of mask_2d.
    mask_2d: boolean array of shape (n, n) = interior grid
    Returns: sparse matrix, flat index array
    """
    n = mask_2d.shape[0]
    L_full, _ = fd_laplacian_2d(n)
    # Scale already uses h from fd_laplacian_2d; rescale to h_s
    h_ref = 1.0 / (n + 1)
    L_full = L_full * h_ref**2 / h**2
    flat_mask = mask_2d.flatten()
    idx = np.where(flat_mask)[0]
    L_sub = L_full[np.ix_(idx, idx)]
    return L_sub, idx

# Define masks (on the n_s x n_s interior grid)
cx, cy, r_outer, r_inner = 0.5, 0.5, 0.45, 0.15
dist = np.sqrt((X_s - cx)**2 + (Y_s - cy)**2)

masks = {
    'Square':  np.ones((n_s, n_s), dtype=bool),
    'Disk':    dist < r_outer,
    'L-shape': ~((X_s > 0.5) & (Y_s > 0.5)),
    'Ring':    (dist < r_outer) & (dist > r_inner),
}

results = {}
for name, mask in masks.items():
    L_sub, idx = mask_laplacian(mask, h_s)
    n_modes = min(9, L_sub.shape[0] - 2)
    vals, vecs = eigsh(L_sub, k=n_modes, which='SM', tol=1e-8)
    order = np.argsort(vals)
    results[name] = {
        'vals': vals[order],
        'vecs': vecs[:, order],
        'idx': idx,
        'mask': mask,
        'area': mask.sum() * h_s**2,
    }
    print(f'{name}: area={mask.sum()*h_s**2:.3f}, λ1={vals[order[0]]:.2f}')

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_52434/1991806073.py:4: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
  T = diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr')


Square: area=0.961, λ1=19.73
Disk: area=0.640, λ1=27.49
L-shape: area=0.721, λ1=37.34
Ring: area=0.567, λ1=96.83


### Visualizing the first 9 eigenmodes per shape

We display the first 9 eigenmodes for each domain shape. Nodal lines (where $u = 0$) are shown in black; blue/red indicate positive/negative regions. Each eigenmode is normalized to $\|u_k\|_{\infty} = 1$ for display.

The eigenmode structure reflects domain geometry: elongated domains have eigenmodes aligned with the long axis, and complex shapes (L-shape, ring) exhibit more intricate patterns.

In [5]:
def plot_eigenmodes(results, name, ax_row):
    res = results[name]
    mask = res['mask']
    idx = res['idx']
    n_modes = len(res['vals'])
    for k in range(min(n_modes, len(ax_row))):
        u_flat = np.zeros(n_s * n_s)
        u_flat[idx] = res['vecs'][:, k]
        u_2d = u_flat.reshape(n_s, n_s)
        u_2d[~mask] = np.nan
        vmax = np.nanmax(np.abs(u_2d))
        ax_row[k].contourf(X_s, Y_s, u_2d, levels=20, cmap='RdBu_r',
                           vmin=-vmax, vmax=vmax)
        try:
            ax_row[k].contour(X_s, Y_s, u_2d, levels=[0], colors='k', linewidths=0.6)
        except Exception:
            pass
        ax_row[k].set_title(f'$\\lambda_{k+1}={res["vals"][k]:.1f}$', fontsize=7)
        ax_row[k].set_aspect('equal')
        ax_row[k].axis('off')

fig, axes = plt.subplots(4, 9, figsize=(18, 8))
for row, name in enumerate(masks.keys()):
    axes[row, 0].set_ylabel(name, fontsize=10, rotation=90, labelpad=5)
    plot_eigenmodes(results, name, axes[row])
plt.suptitle('First 9 Laplacian eigenmodes', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('eigenmodes.png', dpi=70, bbox_inches='tight')
plt.close()

### Weyl's law verification

Weyl's asymptotic law states that for large $k$:
$$\lambda_k \sim \frac{4\pi k}{|\Omega|}$$

This means the counting function $N(\lambda) = \#\{k : \lambda_k \leq \lambda\}$ satisfies:
$$N(\lambda) \sim \frac{|\Omega|}{4\pi} \lambda$$

We plot the numerically computed eigenvalues $\lambda_k$ vs. $k$ for each domain and overlay the Weyl prediction. The slope reveals the domain area.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for (name, res), col in zip(results.items(), colors):
    ks = np.arange(1, len(res['vals']) + 1)
    axes[0].plot(ks, res['vals'], 'o-', label=name, color=col, markersize=5)
    # Weyl prediction
    weyl = 4 * np.pi * ks / res['area']
    axes[0].plot(ks, weyl, '--', color=col, alpha=0.5)

axes[0].set_xlabel('Mode index $k$')
axes[0].set_ylabel('$\\lambda_k$')
axes[0].set_title("Eigenvalue spectra (solid) vs. Weyl's law (dashed)")
axes[0].legend()

# Plot normalized spectra: lambda_k * |Omega| / (4 pi k) should -> 1
for (name, res), col in zip(results.items(), colors):
    ks = np.arange(1, len(res['vals']) + 1)
    ratio = res['vals'] * res['area'] / (4 * np.pi * ks)
    axes[1].plot(ks, ratio, 'o-', label=name, color=col, markersize=5)
axes[1].axhline(1.0, color='k', linestyle='--', label='Weyl limit')
axes[1].set_xlabel('Mode index $k$')
axes[1].set_ylabel('$\\lambda_k |\\Omega| / (4\\pi k)$')
axes[1].set_title("Weyl's law convergence")
axes[1].legend()

plt.tight_layout()
plt.savefig('weyl_law.png', dpi=80, bbox_inches='tight')
plt.close()

### Interactive eigenmode browser

Use the dropdown and slider below to browse eigenmodes across domain shapes. The first eigenfunction (Fiedler vector-like mode) reveals the main axis of the domain.

### Static snapshot

In [7]:
STATIC_SNAPSHOT = True
if STATIC_SNAPSHOT:
    fig, axes = plt.subplots(4, 9, figsize=(18, 8))
    for row, name in enumerate(masks.keys()):
        axes[row, 0].set_ylabel(name, fontsize=9)
        plot_eigenmodes(results, name, axes[row])
    plt.suptitle('Laplacian eigenmodes: Square / Disk / L-shape / Ring', fontsize=11)
    plt.tight_layout()
    plt.savefig('snippet.png', dpi=100, bbox_inches='tight')
    plt.close()

## Takeaways

- The Laplacian spectrum encodes **shape information**: different domains with the same area can have different eigenvalue sequences.
- The **FD discretization** converges to the analytical spectrum; errors decrease as $O(h^2)$.
- Eigenmodes of the rectangle are products of 1D sines, with degeneracies when $m^2 + n^2$ coincides for different $(m,n)$ pairs.
- **Weyl's law** provides a direct link between eigenvalue growth rate and domain area: $\lambda_k \sim 4\pi k / |\Omega|$.
- The first eigenfunction has no nodal lines and is always positive; higher modes exhibit increasingly complex nodal patterns.
- Isospectral (but non-isometric) domains exist (Gordon-Webb-Wolpert, 1992), confirming one cannot always hear the shape of a drum.

## Bibliography

- M. Kac, *Can one hear the shape of a drum?*, American Mathematical Monthly, 73(4):1–23, 1966.
- H. Weyl, *Das asymptotische Verteilungsgesetz der Eigenwerte linearer partieller Differentialgleichungen*, Mathematische Annalen, 71:441–479, 1912.
- C. Gordon, D. Webb, S. Wolpert, *Isospectral plane domains and surfaces via Riemannian orbifolds*, Inventiones Mathematicae, 110:1–22, 1992.
- S. Rosenberg, *The Laplacian on a Riemannian Manifold*, Cambridge University Press, 1997.
- B. Levy, *Laplace-Beltrami eigenfunctions: Towards an algorithm that understands geometry*, IEEE Shape Modeling International, 2006.